In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

session = get_active_session()

session.sql("USE DATABASE NHANES_DB").collect()
session.sql("USE SCHEMA RAW").collect()


df = session.table("NHANES_FINAL").to_pandas()

print(df.shape)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumn Names:\n", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nBasic Stats:\n", df.describe())

In [ ]:
# Fix hypertension flag — set to NULL when BP is missing
df['HYPERTENSION_FLAG'] = df.apply(
    lambda row: None if pd.isnull(row['SYSTOLIC_BP']) 
    else row['HYPERTENSION_FLAG'], axis=1
)

print(df['HYPERTENSION_FLAG'].value_counts(dropna=False))

In [ ]:
# Drop rows where BP is null
df_clean = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP'])

# Define diet columns (only what exists in your table)
diet_cols = ['SODIUM_MG', 'FIBER_G', 'SUGAR_G', 'CALORIES', 'PROTEIN_G', 'FAT_G']

# Fill dietary nulls with median
df_clean[diet_cols] = df_clean[diet_cols].fillna(df_clean[diet_cols].median())

# Fill BMI nulls with median
df_clean['BMI'] = df_clean['BMI'].fillna(df_clean['BMI'].median())

print("Clean shape:", df_clean.shape)
print("\nRemaining nulls:\n", df_clean.isnull().sum())

In [ ]:
df_clean['CARBS_G'] = None  # not available, skip
df_clean['PROTEIN_PCT'] = (df_clean['PROTEIN_G'] * 4 / df_clean['CALORIES']) * 100
df_clean['FAT_PCT'] = (df_clean['FAT_G'] * 9 / df_clean['CALORIES']) * 100
df_clean['CARBS_PCT'] = 100 - df_clean['PROTEIN_PCT'] - df_clean['FAT_PCT']

In [ ]:
print(df_clean[['PROTEIN_PCT','FAT_PCT','CARBS_PCT']].describe())

## Basic EDA for the Project

In [ ]:
# BMI Distribution
df_clean['BMI_CATEGORY'].value_counts().plot(kind='bar', color='steelblue')
plt.title('BMI Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Hypertension Rate
df_clean['HYPERTENSION_FLAG'].value_counts().plot(kind='pie', 
        labels=['No','Yes'], autopct='%1.1f%%', colors=['#4CAF50','#F44336'])
plt.title('Hypertension Rate')
plt.show()

In [ ]:
# Sodium Distribution
df_clean['SODIUM_MG'].plot(kind='hist', bins=50, color='steelblue')
plt.axvline(2300, color='red', linestyle='--', label='2300mg Limit')
plt.title('Sodium Intake')
plt.legend()
plt.show()

print(f"{(df_clean['SODIUM_MG'] > 2300).mean()*100:.1f}% exceed 2300mg limit")

In [ ]:
# Macro Split
df_clean[['PROTEIN_PCT','FAT_PCT','CARBS_PCT']].mean().plot(kind='pie', autopct='%1.1f%%')
plt.title('Macro Split')
plt.show()

In [ ]:
# Sodium vs Hypertension
df_clean.boxplot(column='SODIUM_MG', by='HYPERTENSION_FLAG')
plt.title('Sodium vs Hypertension')
plt.show()

In [ ]:
# BMI by Age Group
df_clean['AGE_GROUP'] = pd.cut(df_clean['AGE'], bins=[18,30,40,50,60,100],
                                labels=['18-30','31-40','41-50','51-60','60+'])
df_clean.boxplot(column='BMI', by='AGE_GROUP')
plt.title('BMI by Age Group')
plt.show()